In [7]:
import os
import sys
import contextlib
import numpy as np
import json
from sklearn.covariance import graphical_lasso
from matplotlib import pyplot as plt

from method import EM_algorithm as em, tlasso
from simulation import simulation_data_generator as dg

COLOR_ASYM = "#2a78d6"
COLOR_EM_DIAG = "#f5239a"
COLOR_GGM = "#e34948"
COLOR_T = "#f5a623"
COLOR_TS = "#23f57e"
COLOR_CHANCE = "#c3c2b7"
 
def make_rho_grid(S, n_rho=50, max_ratio = 1, min_ratio=0.05):
    """Log-spaced, DESCENDING grid on [min_ratio * rho_max, rho_max].
 
    Log spacing because edge count is roughly geometric in rho: linear
    spacing wastes most points in the dense end where the ROC curve barely
    moves. Descending so warm starts run sparse -> dense, which is the
    numerically stable direction when p > n.
    """
    A = np.abs(np.asarray(S, dtype=float)).copy()
    np.fill_diagonal(A, 0.0)
    rmax = float(A.max())
    if not np.isfinite(rmax) or rmax <= 0:
        raise ValueError("rho_max is not positive; check the input matrix.")
    return np.logspace(np.log10(max_ratio * rmax), np.log10(min_ratio * rmax), n_rho)

def edge_confusion(Theta_hat, true_pos_mask, true_neg_mask, tol=1e-8):
    iu = np.triu_indices(Theta_hat.shape[0], k=1)
    est_edges = np.abs(Theta_hat[iu]) > tol
    tp_rate = np.sum(est_edges & true_pos_mask) / true_pos_mask.sum()
    fp_rate = np.sum(est_edges & true_neg_mask) / true_neg_mask.sum()
    return fp_rate, tp_rate


@contextlib.contextmanager
def stream_to(path, also_stderr=True):
    """Send print()/warnings to `path` instead of the cell output.

    buffering=1 is the point: line buffering means the file fills as the run
    goes, so `Get-Content <path> -Wait` follows it live. On an exception the
    streams are restored before the traceback renders, so errors still show
    up in the notebook.
    """
    f = open(path, "w", buffering=1, encoding="utf-8")
    old_out, old_err = sys.stdout, sys.stderr
    sys.stdout = f
    if also_stderr:
        sys.stderr = f
    try:
        yield f
    finally:
        sys.stdout, sys.stderr = old_out, old_err
        f.close()


def roc_curve_em(
    Y, rho_grid, algorithm, true_pos_mask, true_neg_mask,
    algorithm_kwargs=None, rho_name="rho", theta_key="Theta",
):
    """Re-run the whole EM at each rho, warm-started, sparsest first.

    `algorithm_kwargs` holds whatever extra arguments the given algorithm takes
    (e.g. nu/n_iter for run_tlasso, n_burn/n_keep for run_em_MWGP); `rho_name`
    and `theta_key` cover algorithms that name the penalty or the returned
    precision matrix differently.

    Each fit is initialized at the previous rho's mu and Theta. Only those two
    are carried: nu/eta are tail-shape nuisance parameters, and handing them
    forward lets nu ratchet down across the sweep until lam = -2/nu - 0.5 is
    negative enough to overflow the Bessel terms in gig_moment.

    These EM objectives are not convex, so warm starting changes the estimator
    and not just the runtime: every point on the curve depends on the whole
    path prefix, and the sweep must stay sparse -> dense for the results to be
    reproducible.
    """
    kwargs = {} if algorithm_kwargs is None else dict(algorithm_kwargs)
    fp = np.empty(len(rho_grid))
    tp = np.empty(len(rho_grid))
    Theta_prev = np.eye(Y.shape[1])
    prev = None
    print(f"Running {algorithm.__name__} over {len(rho_grid)} rho values...")
    for i in range(len(rho_grid)):
        print(f"  rho={rho_grid[i]:.5f} ({i+1}/{len(rho_grid)})")

        res = algorithm(
            Y, **{rho_name: rho_grid[i]}, **kwargs,
            **({"init": prev} if prev is not None else {}),
        )
        Theta_hat = res[theta_key]
        if np.all(np.isfinite(Theta_hat)):
            prev = {"mu": res["mu"], "Theta": Theta_hat}

        Theta_prev = Theta_hat
        fp[i], tp[i] = edge_confusion(Theta_hat, true_pos_mask, true_neg_mask)
        print(fp[i], tp[i])
    return fp, tp


def roc_curve_glasso(Y, rho_grid, true_pos_mask, true_neg_mask, glasso_kwargs=None):
    """roc_curve_full_em analogue for the naive Gaussian glasso baseline.

    sklearn's graphical_lasso takes an empirical covariance rather than Y,
    names the penalty `alpha`, and returns (covariance, precision), so it needs
    its own loop.
    """
    kwargs = {} if glasso_kwargs is None else dict(glasso_kwargs)
    S = np.cov(Y, rowvar=False) + 1e-10 * np.eye(Y.shape[1])
    fp = np.empty(len(rho_grid))
    tp = np.empty(len(rho_grid))
    
    p = Y.shape[1]
    Theta_prev = np.eye(p)
    print(f"Running graphical_lasso over {len(rho_grid)} rho values...")
    for i in range(len(rho_grid)):
        print(f"  rho={rho_grid[i]:.5f} ({i+1}/{len(rho_grid)})")
        try:
            _, Theta_hat = graphical_lasso(S + rho_grid[i] * np.eye(p), alpha=rho_grid[i], **kwargs)
        except Exception:
            Theta_hat = Theta_prev
        Theta_prev = Theta_hat
        fp[i], tp[i] = edge_confusion(Theta_hat, true_pos_mask, true_neg_mask)
        
    
    return fp, tp


def auc_from_curve(fp, tp):
    order = np.argsort(fp)
    fp_sorted = np.concatenate([[0.0], fp[order], [1.0]])
    tp_sorted = np.concatenate([[0.0], tp[order], [1.0]])
    return np.trapezoid(tp_sorted, fp_sorted)

In [8]:
filename = ""
num_simulations=1
p=20
n=2000
num_rho=30

In [9]:
Theta_true = dg.make_true_theta(p)

iu = np.triu_indices(p, k=1)
true_pos_mask = Theta_true[iu] != 0
true_neg_mask = ~true_pos_mask

theoretical_rho = np.sqrt(np.log(p) / n)
# make a evenly spaced grid centered at theoretical rho with half length width

print(f"theoretical rho = sqrt(log({p}) / {n}) = {theoretical_rho:.5g}")

theoretical rho = sqrt(log(20) / 2000) = 0.038702


In [10]:
fp_mwgp = np.empty((num_simulations, num_rho))
tp_mwgp = np.empty((num_simulations, num_rho))
fp_em_diag = np.empty((num_simulations, num_rho))
tp_em_diag = np.empty((num_simulations, num_rho))
fp_ggm = np.empty((num_simulations, num_rho))
tp_ggm = np.empty((num_simulations, num_rho))
fp_t = np.empty((num_simulations, num_rho))
tp_t = np.empty((num_simulations, num_rho))
fp_ts = np.empty((num_simulations, num_rho))
tp_ts = np.empty((num_simulations, num_rho))
auc_mwgp = np.empty(num_simulations)
auc_em_diag = np.empty(num_simulations)
auc_ggm = np.empty(num_simulations)
auc_t = np.empty(num_simulations)
auc_ts = np.empty(num_simulations)

In [ ]:
# Output goes to roc_run.log so it does not flood the cell.
# Follow it live:  Get-Content roc_run.log -Wait -Tail 20
with stream_to("roc_run.log"):
    for sim in range(num_simulations):
        print(f"Replicate {sim + 1}/{num_simulations}")

        # Generate data from an independent model (noisy skewed Gaussian)
        Y = dg.simulate_contaminated_normal_data(n, p, Theta_true)
        S = np.cov(Y, rowvar=False)
        rho_grid = make_rho_grid(S, n_rho = num_rho)
        print(f"rho grid: {rho_grid}")

        # Alternative t-distribution model (run_tstar_varlasso)
        fp_ts[sim], tp_ts[sim] = roc_curve_em(
            Y, rho_grid, tlasso.run_tstar_varlasso, true_pos_mask, true_neg_mask,
            algorithm_kwargs={"n_iter": 200, "verbose": False}
        )
        auc_ts[sim] = auc_from_curve(fp_ts[sim], tp_ts[sim])
    
        # Asymmetric Alternative t-distribution model (EM_DIAGONAL)
        fp_em_diag[sim], tp_em_diag[sim] = roc_curve_em(
            Y, rho_grid, em.run_em_diagonal, true_pos_mask, true_neg_mask,
            algorithm_kwargs={"n_iter": 200, "verbose": False}
        )
        auc_em_diag[sim] = auc_from_curve(fp_em_diag[sim], tp_em_diag[sim])
    
        # Classical t-distribution model (run_tlasso)
        fp_t[sim], tp_t[sim] = roc_curve_em(
            Y, rho_grid, tlasso.run_tlasso, true_pos_mask, true_neg_mask,
            algorithm_kwargs={"n_iter": 200, "verbose": False}
        )
        auc_t[sim] = auc_from_curve(fp_t[sim], tp_t[sim])
    
        # Asymmetric Alternative t-distribution model (EM_MWGP)
        fp_mwgp[sim], tp_mwgp[sim] = roc_curve_em(
            Y, rho_grid, em.run_em_MWGP, true_pos_mask, true_neg_mask,
            algorithm_kwargs={"n_iter": 50, 
                                "verbose": True, 
                                "warning": True, 
                                "mcmc_samples": 100,
                                "mcmc_thin": 1,
                                "mcmc_warmup": 10,
                                "proposal": "gig"}
        )
        auc_mwgp[sim] = auc_from_curve(fp_mwgp[sim], tp_mwgp[sim])

        # Naive Gaussian graphical lasso baseline
        fp_ggm[sim], tp_ggm[sim] = roc_curve_glasso(
            Y, rho_grid, true_pos_mask, true_neg_mask,
            glasso_kwargs={"max_iter": 2000, "verbose": False}
        )
        auc_ggm[sim] = auc_from_curve(fp_ggm[sim], tp_ggm[sim])


In [ ]:
# Average the ROC curves across replicates and compute mean AUCs
fp_mwgp_mean, tp_mwgp_mean = fp_mwgp.mean(axis=0), tp_mwgp.mean(axis=0)
fp_em_diag_mean, tp_em_diag_mean = fp_em_diag.mean(axis=0), tp_em_diag.mean(axis=0)
fp_ggm_mean, tp_ggm_mean = fp_ggm.mean(axis=0), tp_ggm.mean(axis=0)
fp_t_mean, tp_t_mean = fp_t.mean(axis=0), tp_t.mean(axis=0)
fp_ts_mean, tp_ts_mean = fp_ts.mean(axis=0), tp_ts.mean(axis=0)

print(f"\n(p={p}, n={n}) over {num_simulations} replicates")
print(f"  Asymmetric model (MWGP): AUC = {auc_mwgp.mean():.3f} "
        f"(SE {auc_mwgp.std(ddof=1) / np.sqrt(num_simulations):.3f})")
print(f"  Asymmetric model (Diagonal): AUC = {auc_em_diag.mean():.3f} "
        f"(SE {auc_em_diag.std(ddof=1) / np.sqrt(num_simulations):.3f})")
print(f"  Classical t-model (TLASSO): AUC = {auc_t.mean():.3f} "
        f"(SE {auc_t.std(ddof=1) / np.sqrt(num_simulations):.3f})")
print(f"  Alternative t-model (TSTAR_VARLASSO): AUC = {auc_ts.mean():.3f} "
        f"(SE {auc_ts.std(ddof=1) / np.sqrt(num_simulations):.3f})")
print(f"  Naive Gaussian glasso (GGM): AUC = {auc_ggm.mean():.3f} "
        f"(SE {auc_ggm.std(ddof=1) / np.sqrt(num_simulations):.3f})")

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1, color=COLOR_CHANCE)
ax.plot(
    fp_mwgp_mean, tp_mwgp_mean, color=COLOR_ASYM, linewidth=2,
    label=f"Asym. model MWGP, avg AUC={auc_mwgp.mean():.3f}",
)
ax.plot(
    fp_em_diag_mean, tp_em_diag_mean, color=COLOR_EM_DIAG, linewidth=2,
    label=f"Asym. diag. model, avg AUC={auc_em_diag.mean():.3f}",
)
ax.plot(
    fp_ggm_mean, tp_ggm_mean, color=COLOR_GGM, linewidth=2, linestyle="-.",
    label=f"Naive GGM, avg AUC={auc_ggm.mean():.3f}",
)
ax.plot(
    fp_t_mean, tp_t_mean, color=COLOR_T, linewidth=2, linestyle="-.",
    label=f"Classical t-model, avg AUC={auc_t.mean():.3f}",
)
ax.plot(
    fp_ts_mean, tp_ts_mean, color=COLOR_TS, linewidth=2, linestyle="-.",
    label=f"Alternative t-model, avg AUC={auc_ts.mean():.3f}",
)
# make_rho_grid is log-spaced on [0.05 * rho_max, rho_max], so the
# theoretical rho sits at no fixed index -- locate it instead. Each
# replicate has its own grid, but rho_max varies only ~1.2x across
# replicates, so using the last one costs well under one index.
i_star = int(np.argmin(np.abs(rho_grid - theoretical_rho)))
for fp_mean, tp_mean, color in (
    (fp_mwgp_mean, tp_mwgp_mean, COLOR_ASYM),
    (fp_ggm_mean, tp_ggm_mean, COLOR_GGM),
    (fp_t_mean, tp_t_mean, COLOR_T),
    (fp_ts_mean, tp_ts_mean, COLOR_TS),
    (fp_em_diag_mean, tp_em_diag_mean, COLOR_EM_DIAG),
):
    ax.plot(fp_mean[i_star], tp_mean[i_star], "o", color=color, zorder=5)
ax.plot([], [], "o", color="#52514e",
        label=r"$\rho=\sqrt{\log p\,/\,n}$" + f" = {theoretical_rho:.3g}")

ax.set_xlim(0, 1)
ax.set_ylim(0, 1.2)
ax.set_xlabel("false positive rate (1 - specificity)")
ax.set_ylabel("true positive rate (sensitivity)")
ax.set_title(f"Precision-matrix support recovery: p={p}, n={n}")
ax.legend(loc="lower right")
fig.tight_layout()

os.makedirs("results/simulations/roc", exist_ok=True)
filename = (
    "results/simulations/roc/roc" if filename is None
    else f"results/simulations/roc/roc_{filename}"
)
fig.savefig(f"{filename}.pdf")

results = {
    "p": p,
    "n": n,
    "rho_grid": rho_grid.tolist(),
    "theoretical_rho": float(theoretical_rho),
    "asym_mwgp": {
        "fp_mean": fp_mwgp_mean.tolist(),
        "tp_mean": tp_mwgp_mean.tolist(),
        "auc_mean": float(auc_mwgp.mean()),
        "auc_se": float(auc_mwgp.std(ddof=1) / np.sqrt(num_simulations)),
    },
    "asym_em_diag": {
        "fp_mean": fp_em_diag_mean.tolist(),
        "tp_mean": tp_em_diag_mean.tolist(),
        "auc_mean": float(auc_em_diag.mean()),
        "auc_se": float(auc_em_diag.std(ddof=1) / np.sqrt(num_simulations)),
    },
    "ggm": {
        "fp_mean": fp_ggm_mean.tolist(),
        "tp_mean": tp_ggm_mean.tolist(),
        "auc_mean": float(auc_ggm.mean()),
        "auc_se": float(auc_ggm.std(ddof=1) / np.sqrt(num_simulations)),
    },
    "t": {
        "fp_mean": fp_t_mean.tolist(),
        "tp_mean": tp_t_mean.tolist(),
        "auc_mean": float(auc_t.mean()),
        "auc_se": float(auc_t.std(ddof=1) / np.sqrt(num_simulations)),
    },
    "ts": {
        "fp_mean": fp_ts_mean.tolist(),
        "tp_mean": tp_ts_mean.tolist(),
        "auc_mean": float(auc_ts.mean()),
        "auc_se": float(auc_ts.std(ddof=1) / np.sqrt(num_simulations)),
    },
}

with open(f"{filename}.json", "w") as f:
    json.dump(results, f)

In [ ]:
# Y = dg.simulate_contaminated_normal_data(n, p, Theta_true)
# S = np.cov(Y, rowvar=False)
# rho_grid = make_rho_grid(S, n_rho = num_rho)
# print(f"rho grid: {rho_grid}")

In [ ]:
# results = em.run_em_diagonal(Y, n_iter = 60, rho = rho_grid[5])